# Downloading a light curve from MAST

Mimir keeps archive access optional and delegates it to Lightkurve. This
notebook shows both:

- the convenient target-name interface; and
- the individual search, download, reduction, and conversion steps.

The cells access the network and may download multiple Kepler quarters.
Install the optional dependencies with:

```bash
python -m pip install "mimir-astro[mast] @ git+https://github.com/nielsenmb/Mimir.git" matplotlib
```


In [ ]:
import matplotlib.pyplot as plt

from mimir import (
    as_timeseries,
    download_lightcurves,
    lightcurve_to_timeseries,
    power_spectrum,
    reduce_lightcurve,
    search_lightcurves,
)

plt.rcParams["figure.figsize"] = (8, 4)


## Convenient target-name interface

Any target identifier understood by Lightkurve can be passed to
`as_timeseries`, `power_spectrum`, or `spectral_window`. Lightkurve search
arguments are nested under `search_kwargs`; the other entries configure
Mimir's download and reduction stages.

Here `numax` is used only to choose a sensible flattening window. It is not a
prior or constraint on the calculated spectrum.


In [ ]:
target = "KIC 8006161"
mast_kwargs = {
    "search_kwargs": {
        "mission": "Kepler",
        "author": "Kepler",
        "exptime": 60,
    },
    "numax": 3500.0,
}

series = as_timeseries(target=target, mast_kwargs=mast_kwargs)


In [ ]:
{
    "samples": series.n_samples,
    "duration (d)": series.duration,
    "cadence (s)": series.cadence * 86400.0,
    "duty cycle": series.duty_cycle,
    "flux unit": series.flux_unit,
}


In [ ]:
fig, ax = plt.subplots()
ax.plot(series.time, series.flux, ".", ms=0.6)
ax.set(
    xlabel=f"Time [{series.time_unit}]",
    ylabel=f"Relative flux [{series.flux_unit}]",
    title=target,
)
plt.show()


Reusing the returned `TimeSeries` avoids downloading the data again when
calculating several products.


In [ ]:
spectrum = power_spectrum(time_series=series, oversampling=2)

fig, ax = plt.subplots()
selection = (spectrum.frequency > 500.0) & (spectrum.frequency < 5000.0)
ax.plot(
    spectrum.frequency[selection],
    spectrum.power_density[selection],
    lw=0.6,
)
ax.set(
    xlabel=f"Frequency [{spectrum.frequency_unit}]",
    ylabel=f"Power density [{series.flux_unit}²/{spectrum.frequency_unit}]",
    title=f"{target}: power-density spectrum",
)
plt.show()


For a one-off calculation, the complete workflow can be written as one call:

```python
spectrum = power_spectrum(
    target="KIC 8006161",
    mast_kwargs={
        "search_kwargs": {
            "mission": "Kepler",
            "author": "Kepler",
            "exptime": 60,
        },
        "numax": 3500.0,
    },
    oversampling=2,
)
```

The explicit `as_timeseries` form is usually preferable in a notebook because
the download is then clearly separated from later numerical work.


## Inspect each archive and reduction stage

Use the lower-level functions when you want to inspect the Lightkurve search
result, select particular quarters or sectors, or change the basic reduction.
The following example downloads only the first matching product.


In [ ]:
search_result = search_lightcurves(
    target,
    mission="Kepler",
    author="Kepler",
    exptime=60,
)
search_result


In [ ]:
selected_result = search_result[:1]
collection = download_lightcurves(selected_result)

reduced_lightcurve = reduce_lightcurve(
    collection,
    outlier_sigma=5.0,
    flatten=True,
    exposure_time=60.0,
    numax=3500.0,
)
single_product_series = lightcurve_to_timeseries(
    reduced_lightcurve,
    ppm=True,
)


In [ ]:
{
    "selected products": len(selected_result),
    "samples": single_product_series.n_samples,
    "duration (d)": single_product_series.duration,
}


Useful controls include:

- `search_kwargs`: `mission`, `author`, `exptime`, `sector`, or `quarter`
  arguments accepted by `lightkurve.search_lightcurve`;
- `download_dir`: the Lightkurve cache location;
- `outlier_sigma=None`: skip outlier removal;
- `flatten=False`: retain long-period trends;
- `flatten_window_length`: set the Lightkurve window directly;
- `ppm=False`: keep the normalized Lightkurve flux scale.

Flattening is a scientific choice. For signals near the low-frequency cutoff,
inspect the unflattened light curve and test that your result is stable under
reasonable changes to the window length.
